# Lesson 2: Introduction to vLLM

[vLLM](https://github.com/vllm-project/vllm) is a high-throughput inference engine designed specifically for serving LLMs. Its key innovations include:

- **PagedAttention** — manages KV cache memory like virtual memory pages, eliminating waste from fragmentation
- **Continuous batching** — dynamically schedules requests so the GPU is always utilized
- **Optimized kernels** — custom CUDA kernels for attention, quantization, and more
- **[Parallelism](https://docs.vllm.ai/en/latest/serving/parallelism_scaling/)** — split models across multiple GPUs seamlessly

In this lesson, we'll use the same **Qwen3-0.6B** model and cover:

1. The `LLM` class for offline inference
2. Chat-style generation
3. The `vllm serve` CLI
4. Key differences from Transformers

## The `LLM` Class

The [`LLM`](https://docs.vllm.ai/en/latest/serving/offline_inference/) class is vLLM's main Python API for offline (batch) inference. Unlike Transformers where you manage tokenization, batching, and device placement yourself, vLLM handles all of this internally.

You just pass in text prompts and get back results.

In [ ]:
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-0.6B"

llm = LLM(model=MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print("Ready for inference!")

## Text Generation

The `generate` method takes raw text prompts (no manual tokenization needed) and returns `RequestOutput` objects containing the generated text and metadata.

Generation is controlled via [`SamplingParams`](https://docs.vllm.ai/en/latest/api/vllm/#vllm.SamplingParams):

In [ ]:
sampling_params = SamplingParams(
    max_tokens=100,
    temperature=0.7,
)

prompts = [
    "The capital of France is",
    "Write a haiku about programming:",
    "The three laws of thermodynamics are: 1.",
    "In a galaxy far far away,",
]

outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"Prompt: {output.prompt!r}")
    print(f"Generated: {output.outputs[0].text!r}\n")

## Chat-style Generation

vLLM also supports chat-style generation via `llm.chat()`. This handles chat template formatting internally, so you just pass in message dicts.

In [ ]:
conversations = [
    [{"role": "user", "content": "What is the capital of France?"}],
    [{"role": "user", "content": "Write a haiku about programming."}],
    [{"role": "user", "content": "What are the three laws of thermodynamics?"}],
    [{"role": "user", "content": "Describe a galaxy far, far away."}],
]

sampling_params = SamplingParams(max_tokens=100, temperature=0.7)

outputs = llm.chat(conversations, sampling_params)

for output in outputs:
    print(f"Conversation: {output.prompt!r}")
    print(f"Generated: {output.outputs[0].text!r}\n")

## Cleaning Up

vLLM's `LLM` class holds the model weights and KV cache in memory. When you're done with offline inference, you should explicitly delete it to free that memory — especially important before starting the server.

In [ ]:
import gc

del llm
gc.collect()

## The `vllm serve` CLI

vLLM provides an [OpenAI-compatible server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/) that exposes the same endpoints we used in Lesson 1 (`POST /v1/chat/completions`, `GET /v1/models`, `GET /health`).

We pass `--enforce-eager` below to skip `torch.compile` and CUDA graph capture — that keeps startup fast for the tutorial, but drop it for real workloads to get vLLM's full performance.

See the [full CLI reference](https://docs.vllm.ai/en/latest/cli/serve/) for all available options.

Let's start it as a background process so we can query it from this notebook.

In [ ]:
import subprocess
import time
import urllib.request

server = subprocess.Popen(["vllm", "serve", MODEL_NAME, "--enforce-eager"])
print(f"Started vllm serve with PID: {server.pid}")

# Wait for the server to be ready
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print("Server is ready!")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start in time")

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "What is a mixture-of-experts model?"}],
    max_tokens=100,
)

print(response.choices[0].message.content)

## Stopping the Server

When you're done, terminate the server process using the PID we captured earlier.

In [ ]:
server.terminate()
server.wait()
print(f"Server (PID {server.pid}) stopped.")

## Transformers vs vLLM

| | Transformers | vLLM |
|---|---|---|
| **Primary use case** | Source of truth for model implementations, research, prototyping, training | High-throughput, low latency serving |
| **Tokenization** | Manual | Automatic |
| **Batching** | Manual (or `generate_batch`) | Always continuous batching |
| **KV cache** | Standard or paged | Always PagedAttention |
| **Model support** | All HF models | Subset (but growing via [Transformers backend](https://docs.vllm.ai/en/latest/models/supported_models/#transformers)!) |
| **Flexibility** | Very high (custom training, PEFT, etc.) | Inference-focused |
| **Throughput** | Good | Excellent |

Both libraries serve different needs and complement each other well. In the next lesson, we'll see how the **Transformers modelling backend** bridges the gap, allowing vLLM to run any compatible Transformers model.